# GAN

This notebook is meant as tutorial for generative adversarial networks as part of the paper "A comparison of generative deep learning methods for multivariate angular simulation". We start by importing some modules.

In [ ]:
import h5py
import numpy as np
import pandas as pd

# The models are implemented in the deep learning framework pytorch
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

from tqdm import tqdm


We will apply our model to the sparse gaussian double Pareto example from the paper. We start by reading in the data. We choose the 10 dimensional case with 10 000 samples:

In [ ]:
with h5py.File("data/sparse_gaussian_data_double_pareto.h5", "r") as h5file:
    print(h5file.keys())
    data = pd.DataFrame(h5file["dataset_n_10000_d_10"][:]).transpose().to_numpy()
    

As we are only interested in the angular part we start by transforming our data into generalized spherical coordinates. Different from the other models for the GANs we do not split the data into training and validation set (see paper):

In [ ]:
def cartesian_to_polar(points):
    """
    Vectorized function to convert multiple points from N-dimensional Cartesian
    coordinates to generalized polar coordinates.

    Parameters:
    points (numpy array): Cartesian coordinates (M points in N-dimensional space).
                          Shape = (M, N)

    Returns:
    numpy array: Polar coordinates for each point.
                 Shape = (M, N), where the first column is the radial distance
                 and the remaining columns are the angles.
    """
    points = np.atleast_2d(points)  # Ensure points are at least 2D
    M, N = points.shape  # M is number of points, N is the dimensionality

    # Step 1: Calculate the radial distance (norm for each point)
    r = np.linalg.norm(points, axis=1)  # Radial distances

    # Step 2: Calculate the angular coordinates
    phi = np.zeros((M, N - 1))
    for i in range(N - 1):
        # Norm of the remaining components starting from index i
        norm = np.linalg.norm(points[:, (i + 1) :], axis=1)
        phi[:, i] = np.arctan2(norm, points[:, i])

    # Special case: The last angle is based on the last two coordinates using arctan2
    phi[:, -1] = np.arctan2(points[:, -1], points[:, -2])

    return r, phi

_, angles_train = cartesian_to_polar(data)

Next we define the model. A GAN consists of both a generator and a discriminator neural network:

In [ ]:
# Generator Network
class Generator(nn.Module):
    def __init__(self, latent_dim, data_dim, num_hidden=2, hidden_dim=64):
        super(Generator, self).__init__()

        self.model = nn.Sequential(
            nn.Linear(latent_dim, hidden_dim),
            nn.ReLU(),
            *(num_hidden * [nn.Linear(hidden_dim, hidden_dim), nn.ReLU(), nn.Dropout(0.2)]),
            nn.Linear(hidden_dim, data_dim),
        )

    def forward(self, z):
        z = self.model(z)
        z[:, :-1] = torch.pi * torch.sigmoid(z[:, :-1])
        z[:, -1] = torch.pi * torch.tanh(z[:, -1])

        return z


# Discriminator Network
class Discriminator(nn.Module):
    def __init__(self, data_dim, num_hidden=2, hidden_dim=64):
        super(Discriminator, self).__init__()
        self.model = nn.Sequential(
            nn.Linear(data_dim, hidden_dim),
            nn.LeakyReLU(0.2),
            *(num_hidden * [nn.Linear(hidden_dim, hidden_dim), nn.LeakyReLU(0.2)]),
            nn.Linear(hidden_dim, 1),
            nn.Sigmoid(),
        )

    def forward(self, x):
        return self.model(x)

We initialize it for our dataset with 10 dimensions. We choose 4 hidden layers with 128 units each.

In [ ]:
generator = Generator(dimension, dimension, 4, 128)
discriminator = Discriminator(dimension, 4, 128)

We check whether a GPU is available and move the model onto there. Also, we convert the training data from numpy to pytorch and also move it onto the GPU if is available. For minibatching we define dataloaders. These progressively output randomly sampled batches of data:

In [ ]:
# Device setup
enable_cuda = True
device = torch.device("cuda" if torch.cuda.is_available() and enable_cuda else "cpu")
vf = vf.to(device)

# Move neural networks to GPU (if available)
generator = generator.to(device)
discriminator = discriminator.to(device)

# Convert data to torch tensors
angles_train = torch.tensor(angles_train).float().to(device)

# DataLoader for training sets
batch_size = 256
train_dataloader = DataLoader(TensorDataset(angles_train), batch_size=batch_size, shuffle=True)

We set a number of training parameters and define the optimizer:

In [ ]:
# Training parameters
n_epochs = 1000 
batch_size = 256
learning_rate = 3e-4

# Loss history
loss_hist = np.array([])
loss_hist_val = np.array([])

# Optimizer
optimizer_G = optim.Adam(generator.parameters(), lr=learning_rate, betas=(0.5, 0.999))
optimizer_D = optim.Adam(discriminator.parameters(), lr=learning_rate, betas=(0.5, 0.999))

Finally, our training loop. This one iterates over the dataset for `n_epochs` iterations (epochs). In each step, we iterate through the training dataloader and:

- Sample random latent noise from a standard normal distribution.
- Generate fake data using the generator.
- Train the discriminator by distinguishing between real and fake data.
- Train the generator by trying to fool the discriminator.

Every 50 epochs, we log the discriminator and generator losses to monitor training progress.

In [ ]:
criterion = nn.BCELoss()    

# Training loop
for epoch in tqdm(range(n_epochs)):  # Iterate through the specified number of epochs
    for real_data_batch in train_dataloader:
        real_data = real_data_batch[0].to(device)  # Move real data batch to the designated device (GPU/CPU)
        batch_size = real_data.size(0)  # Get batch size (useful for last batch, which may be smaller)

        # Define labels for real and fake samples
        real_labels = torch.ones((batch_size, 1)).to(device)  # Label real data as 1
        fake_labels = torch.zeros((batch_size, 1)).to(device)  # Label fake data as 0

        # Train the Discriminator (D)
        z = torch.randn(batch_size, dimension).to(device)  # Sample random noise for fake data
        fake_data = generator(z)  # Generate fake data using the generator

        optimizer_D.zero_grad()  # Zero out gradients before backward pass
        real_loss = criterion(discriminator(real_data), real_labels)  # Loss for real data
        fake_loss = criterion(discriminator(fake_data.detach()), fake_labels)  # Loss for fake data (detach prevents G update here)
        d_loss = real_loss + fake_loss  # Total discriminator loss
        d_loss.backward()  # Compute gradients
        optimizer_D.step()  # Update discriminator weights

        # Train the Generator (G)
        z = torch.randn(batch_size, dimension).to(device)  # Sample new random noise
        fake_data = generator(z)  # Generate fake data

        optimizer_G.zero_grad()  # Zero out gradients before backward pass
        g_loss = criterion(discriminator(fake_data), real_labels)  # Flip labels (trick D into thinking fake is real)
        g_loss.backward()  # Compute gradients
        optimizer_G.step()  # Update generator weights

    # Logging training progress every 50 epochs
    if epoch % 50 == 0 and print_loss:
        print(f"[Epoch {epoch}/{n_epochs}] [D loss: {d_loss.item():.4f}] [G loss: {g_loss.item():.4f}]")

After the model is trained we can sample from it, by sampling from teh generator:

In [ ]:
noise = torch.randn(int(1e6), dimension).to(device)
samples = generator(noise).detach().cpu().numpy()

`samples` now contains samples from the generator in generalized spherical coordinates. We can of course transform them to Euclidean coordinates:

In [ ]:
def polar_to_cartesian(r, phi):
    """
    Vectorized function to convert multiple points from N-dimensional polar coordinates
    back to Cartesian coordinates.

    Parameters:
    polar_coords (numpy array): Polar coordinates (M points in N-dimensional space).
                                Shape = (M, N), where the first column is the radial distance
                                and the remaining columns are the angles.

    Returns:
    numpy array: Cartesian coordinates for each point.
                 Shape = (M, N), where each row corresponds to a point in Cartesian coordinates.
    """
    phi = np.atleast_2d(phi)  # Ensure at least 2D
    M, N = phi.shape  # M is number of points, N is the dimensionality (including radial distance)
    N = N + 1

    # Initialize output array for Cartesian coordinates
    cartesian_coords = np.zeros((M, N))

    # Step 1: Compute the first coordinate x_1
    cartesian_coords[:, 0] = r * np.cos(phi[:, 0])

    # Step 2: Compute the remaining coordinates
    sin_product = np.sin(phi[:, 0])  # Cumulative sin product starts with sin(theta_1)
    for i in range(1, N - 1):
        cartesian_coords[:, i] = r * sin_product * np.cos(phi[:, i])
        sin_product *= np.sin(phi[:, i])  # Update sin product cumulatively

    # Step 3: Compute the last coordinate x_N
    cartesian_coords[:, -1] = r * sin_product  # Last component involves only sin product

    return cartesian_coords


samples_euclidean = polar_to_cartesian(np.ones_like(samples), samples)